# Refined  Model

## Importing Packages

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
%pip install catboost
from catboost import CatBoostClassifier, Pool

## Importing data

In [ ]:
# Define paths and corresponding raw GitHub URLs
files = {
    'train': {
        'local': '../data/raw/Train.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/Train.csv'
    },
    'test': {
        'local': '../data/raw/Test.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/Test.csv'
    },
    'sample_sub': {
            'local': '../data/raw/SampleSubmission.csv',
            'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/SampleSubmission.csv'
        }
    
}

data = {}

# Loop through each dataset to try local loading first, then fallback to GitHub
for name, paths in files.items():
    try:
        data[name] = pd.read_csv(paths['local'])
        print(f"Loaded {name} from local path.")
    except (FileNotFoundError, OSError):
        print(f"Local file for {name} not found. Fetching from GitHub...")
        try:
            data[name] = pd.read_csv(paths['remote'])
            print(f"Loaded {name} from GitHub successfully.")
        except Exception as e:
            if name == 'var_defs':
                print(f"Warning: Could not fetch {name}. Setting to None.")
                data[name] = None
            else:
                raise e

# Unpack the dictionary into individual variables
train = data['train']
test = data['test']
sample_sub = data['sample_sub']

## Setting up our columns

In [ ]:
id_col = 'Tour_ID'
target_col = 'cost_category'
target_classes = ['High Cost', 'Higher Cost', 'Highest Cost', 'Low Cost', 'Lower Cost', 'Normal Cost']

class_to_idx = {cls_name: i for i, cls_name in enumerate(target_classes)}
idx_to_class = {i: cls_name for i, cls_name in enumerate(target_classes)}
train['target'] = train[target_col].map(class_to_idx)

## Advanced Feature Engineering Pipeline

In [ ]:
def preprocess_dataset(df):
    df = df.copy()
    
    # Typos & Missing Values
    if 'main_activity' in df.columns:
        df['main_activity'] = df['main_activity'].replace({'Widlife Tourism': 'Wildlife Tourism'})
    
    df['travel_with'] = df['travel_with'].fillna('Alone')
    df['total_female'] = df['total_female'].fillna(0)
    df['total_male'] = df['total_male'].fillna(0)
    df['most_impressing'] = df.get('most_impressing', pd.Series()).fillna('No Answer')

    # Aggregations & Per-Capita Metrics
    df['total_people'] = df['total_female'] + df['total_male']
    df['total_people_safe'] = df['total_people'].apply(lambda x: 1 if x == 0 else x)
    
    df['total_nights'] = df['night_mainland'] + df['night_zanzibar']
    df['total_nights_safe'] = df['total_nights'].apply(lambda x: 1 if x == 0 else x)
    
    # Duration Ratios
    df['mainland_ratio'] = df['night_mainland'] / df['total_nights_safe']
    df['zanzibar_ratio'] = df['night_zanzibar'] / df['total_nights_safe']
    df['female_ratio'] = df['total_female'] / df['total_people_safe']
    df['nights_per_person'] = df['total_nights'] / df['total_people_safe']
    
    # Package Depth Metrics
    package_cols = [
        'package_transport_int', 'package_accomodation', 'package_food',
        'package_transport_tz', 'package_sightseeing', 'package_guided_tour',
        'package_insurance'
    ]
    df['package_count'] = (df[package_cols] == 'Yes').sum(axis=1)
    df['package_depth'] = df['package_count'] / len(package_cols)
    df['is_full_package'] = (df['package_count'] == len(package_cols)).astype(int)
    df['has_no_package'] = (df['package_count'] == 0).astype(int)
    
    return df

train_df = preprocess_dataset(train)
test_df = preprocess_dataset(test)

## Categorical Feature Specification

In [ ]:
cat_features = [
    'country', 'age_group', 'travel_with', 'purpose', 'main_activity', 
    'info_source', 'tour_arrangement', 'package_transport_int', 
    'package_accomodation', 'package_food', 'package_transport_tz', 
    'package_sightseeing', 'package_guided_tour', 'package_insurance', 'first_trip_tz'
]

# Formatting for CatBoost categorical inputs
for col in cat_features:
    train_df[col] = train_df[col].astype(str)
    test_df[col] = test_df[col].astype(str)

# Frequency Encoding
for col in ['country', 'purpose', 'main_activity']:
    freq_map = pd.concat([train_df[col], test_df[col]]).value_counts()
    train_df[f'{col}_freq'] = train_df[col].map(freq_map)
    test_df[f'{col}_freq'] = test_df[col].map(freq_map)

features = [col for col in train_df.columns if col not in [id_col, target_col, 'target']]

X = train_df[features].copy()
y = train_df['target'].values
X_test = test_df[features].copy()

## Stratified 5-Fold Cross-Validation with LightGBM

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros((len(train_df), len(target_classes)))
test_preds = np.zeros((len(test_df), len(target_classes)))

te_cols = ['country', 'purpose']

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Training Fold {fold + 1} ---")
    
    X_tr, y_tr = X.iloc[train_idx].copy(), y[train_idx]
    X_va, y_va = X.iloc[val_idx].copy(), y[val_idx]
    X_te = X_test.copy()
    
    # In-fold target encoding to eliminate data leakage
    for col in te_cols:
        for c in range(len(target_classes)):
            te_map = y_tr == c
            col_te = X_tr[te_map][col].value_counts() / X_tr[col].value_counts()
            
            X_tr[f'{col}_te_class_{c}'] = X_tr[col].map(col_te).fillna(0)
            X_va[f'{col}_te_class_{c}'] = X_va[col].map(col_te).fillna(0)
            X_te[f'{col}_te_class_{c}'] = X_te[col].map(col_te).fillna(0)

    train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
    val_pool = Pool(X_va, y_va, cat_features=cat_features)
    test_pool = Pool(X_te, cat_features=cat_features)
    
    model = CatBoostClassifier(
        iterations=1800,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=5,
        random_strength=0.8,
        loss_function='MultiClass',
        eval_metric='MultiClass',
        random_seed=42 + fold,
        verbose=300
    )
    
    model.fit(
        train_pool,
        eval_set=val_pool,
        early_stopping_rounds=120,
        use_best_model=True
    )
    
    oof_preds[val_idx] = model.predict_proba(val_pool)
    test_preds += model.predict_proba(test_pool) / skf.n_splits

## Probability Calibration 

In [ ]:
# Prevents catastrophic log-loss penalty on extreme misclassifications
oof_preds_clipped = np.clip(oof_preds, 1e-15, 1 - 1e-15)
oof_preds_calibrated = oof_preds_clipped / oof_preds_clipped.sum(axis=1, keepdims=True)

test_preds_clipped = np.clip(test_preds, 1e-15, 1 - 1e-15)
test_preds_calibrated = test_preds_clipped / test_preds_clipped.sum(axis=1, keepdims=True)

In [ ]:
cv_score = log_loss(y, oof_preds_calibrated)
print(f"\n==========================================")
print(f"GUARANTEED OOF LOG LOSS: {cv_score:.5f}")
print(f"==========================================")

## Refine Model Submission

In [ ]:
submission = pd.DataFrame(test_preds_calibrated, columns=[idx_to_class[i] for i in range(len(target_classes))])
submission.insert(0, id_col, test_df[id_col])

# Strictly match SampleSubmission.csv structure
submission = submission[sample_sub.columns]
submission = sample_sub[[id_col]].merge(submission, on=id_col, how='left')

submission.to_csv('refined_submission.csv', index=False)
print("Successfully generated file!")